# CDT vs S&P 500 vs core (Colombia, 2000–2026)

Notebook reproducible: usa la caché `cache/*.parquet` (o `--refresh` en `main.py`). Análisis educativo, no es recomendación.

**Pregunta:** *¿Para qué invierto en bolsa si el CDT me da 13 % y el S&P 10 %?*

In [1]:
import pandas as pd, json
from IPython.display import Image, display
import main
r = main.build()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print('Corte:', r['res']['fecha_corte'], '| TRM:', r['res']['trm_corte'])

Corte: 2026-11-10 | TRM: 3269.88


## 1. Fuentes descargadas

In [2]:
pd.read_csv('output/fuentes_descarga.csv')[['serie','desde','hasta','filas','descargado_bogota']]

,serie,desde,hasta,filas,descargado_bogota
0,DTF 90 días diaria (vigencia),1984-01-16,2026-11-15,15645,2026-11-11 14:03
1,DTF 90 días mensual,1986-01-01,2026-10-01,490,2026-11-11 14:03
2,TRM COP/USD,1967-03-22,2026-11-12,21786,2026-11-11 14:04
3,IPC Colombia (DANE),2003-01-01,2026-10-01,286,2026-11-11 14:04
4,"IPC Colombia 1999-11..2002-12 (OCDE, fuente DANE)",1999-11-01,2002-12-01,38,2026-11-11 14:04
5,CPI EE.UU. CPIAUCSL (BLS vía FRED),1999-01-01,2026-10-01,334,2026-11-11 14:04
6,Yahoo ^SP500TR,1999-12-01,2026-11-10,6777,2026-11-11 14:04
7,Yahoo ^GSPC,1999-12-01,2026-11-10,6777,2026-11-11 14:04
8,Yahoo SPY,1999-12-01,2026-11-10,6777,2026-11-11 14:04
9,Yahoo SPYG,2000-10-02,2026-11-10,6566,2026-11-11 14:04


## 2. Retornos año a año (%)

In [3]:
cols=['dtf_promedio_ea','cdt_cop','sp500tr_usd','sp500tr_cop','core_cop','inflacion_col','devaluacion_trm','cdt_real_cop','sp500tr_real_cop']
(r['ar'][cols]*100).round(1)

,dtf_promedio_ea,cdt_cop,sp500tr_usd,sp500tr_cop,core_cop,inflacion_col,devaluacion_trm,cdt_real_cop,sp500tr_real_cop
año,,,,,,,,,
2000,12.1,12.9,-9.1,8.1,NaN,8.8,19.0,3.9,-0.6
2001,12.4,12.7,-11.9,-9.4,NaN,7.6,2.8,4.7,-15.9
2002,8.9,9.5,-22.1,-2.6,NaN,7.0,25.0,2.3,-9.0
2003,7.8,7.8,28.7,24.8,33.7,6.5,-3.0,1.2,17.2
2004,7.8,7.9,10.9,-4.6,-10.9,5.5,-14.0,2.3,-9.6
2005,7.0,7.2,4.9,0.3,5.2,4.8,-4.4,2.2,-4.4
2006,6.3,6.2,15.8,13.5,10.7,4.5,-2.0,1.7,8.6
2007,8.0,7.7,5.5,-5.1,3.9,5.7,-10.0,1.9,-10.2
2008,9.7,9.3,-37.0,-29.8,-33.2,7.7,11.4,1.5,-34.8


In [4]:
r['sm']

,acumulado,cagr,volatilidad_anual,peor_año,mejor_año,años_negativos,max_drawdown
CDT COP,5.216707,0.070393,0.030454,2.0% (2021),13.4% (2023),0 de 26,0.0
S&P 500 TR USD,7.351939,0.082223,0.17967,-37.0% (2008),32.4% (2013),6 de 26,-0.552502
S&P 500 TR COP,13.574808,0.104891,0.193918,-29.8% (2008),49.3% (2021),6 de 26,-0.454735
CDT USD,2.562418,0.048433,0.147785,-20.7% (2015),42.7% (2023),11 de 26,-0.465344
Core USD (desde 2003),18.722343,0.134822,0.191824,-40.0% (2008),39.4% (2009),5 de 22,-0.536599
Core COP (desde 2003),21.062316,0.140232,0.206018,-33.2% (2008),48.4% (2024),4 de 22,-0.465808
CDT COP 2003-hoy,3.368717,0.064539,NaN,NaN,NaN,NaN,0.0
S&P 500 TR COP 2003-hoy,13.802763,0.121094,NaN,NaN,NaN,NaN,-0.454735
S&P 500 TR USD 2003-hoy,12.232752,0.115775,NaN,NaN,NaN,NaN,-0.552502
Core COP 2003-hoy,21.062316,0.140232,NaN,NaN,NaN,NaN,-0.465808


In [5]:
display(Image('output/cdt_vs_sp500_retornos_anuales.png'))

(Imagen omitida en la versión de GitHub: ver output/*.png)


## 3. Subperiodos de estrés

In [6]:
(r['sub']*100).round(1)

,CDT COP,S&P 500 TR USD,S&P 500 TR COP,CDT USD,Core USD (desde 2003),Core COP (desde 2003)
2000-2002 (puntocom),39.3,-37.6,-4.6,-8.9,NaN,NaN
2008 (crisis financiera),9.3,-37.0,-29.8,-1.8,-40.0,-33.2
"2020 (covid, año)",3.8,18.4,24.0,-0.9,27.4,33.4
feb-mar 2020 (caída covid),0.4,-33.8,-20.8,-16.1,-31.2,-17.7
2022 (tasas/inflación),7.1,-18.1,-1.1,-11.4,-21.4,-5.1


## 4. Simulación USD 1.000 + USD 200/mes

In [7]:
d=r['dca'].copy()
for k in ['xirr','max_dd_patrimonio','max_caida_vs_aportado','xirr_real_cop']: d[k]=(d[k]*100).round(2)
d['aportado']=d['aportado'].round(0); d['valor_final']=d['valor_final'].round(0); d['multiplo']=d['multiplo'].round(2)
d.drop(columns=['aportado_pesos_oct2026'])

,ventana,moneda,estrategia,aportado,valor_final,multiplo,xirr,max_dd_patrimonio,max_caida_vs_aportado,xirr_real_cop
0,2000-hoy,COP,CDT (DTF 90d),181742008.0,4.495637e+08,2.47,6.66,0.00,0.00,1.42
1,2000-hoy,COP,S&P 500 TR,181742008.0,1.302919e+09,7.17,13.17,-37.72,-31.34,7.69
2,2000-hoy,USD,CDT (DTF 90d),65400.0,1.374860e+05,2.10,5.04,-38.34,-9.84,NaN
3,2000-hoy,USD,S&P 500 TR,65400.0,3.984610e+05,6.09,11.40,-47.89,-37.12,NaN
4,2000-hoy,COP,CDT (DTF 90d) ret4,181742008.0,4.316885e+08,2.38,6.39,0.00,0.00,1.16
5,2000-hoy,COP,CDT (DTF 90d) ret_hist,181742008.0,4.275542e+08,2.35,6.32,0.00,0.00,1.10
6,2003-hoy,COP,CDT (DTF 90d),163728816.0,3.540641e+08,2.16,6.61,0.00,0.00,1.35
7,2003-hoy,COP,S&P 500 TR,163728816.0,1.039105e+09,6.35,14.31,-33.47,-34.60,8.82
8,2003-hoy,COP,Core,163728816.0,1.365979e+09,8.34,16.14,-38.70,-34.86,10.58
9,2003-hoy,USD,CDT (DTF 90d),57400.0,1.082800e+05,1.89,4.97,-36.24,-15.73,NaN


In [8]:
print('Tramos con CDT adelante (DCA 2000, COP):', r['res']['dca_tramos_cdt_adelante'])
display(Image('output/cdt_vs_sp500_dca_cop.png')); display(Image('output/cdt_vs_sp500_dca_usd.png'))

Tramos con CDT adelante (DCA 2000, COP): ['2000-02..2000-02', '2000-10..2014-01', '2014-03..2014-07']


## 5. Ventanas rodantes

In [9]:
r['rr'].round(3)

,ventana_años,comparación,n_ventanas,pct_gana_bolsa,peor_cagr_bolsa,mejor_cagr_bolsa,peor_cagr_cdt,mejor_cagr_cdt,pct_bolsa_negativa,primera_ventana_fin,ultima_ventana_fin
0,5,S&P 500 TR COP vs CDT COP,264,0.614,-0.076,0.289,0.040,0.101,0.231,2004-12,2026-11
1,5,Core COP vs CDT COP,224,0.728,-0.068,0.294,0.040,0.097,0.170,2008-04,2026-11
2,5,S&P 500 TR USD vs CDT USD,264,0.614,-0.066,0.230,-0.069,0.184,0.102,2004-12,2026-11
3,10,S&P 500 TR COP vs CDT COP,204,0.711,-0.037,0.255,0.045,0.088,0.078,2009-12,2026-11
4,10,Core COP vs CDT COP,164,0.902,0.030,0.261,0.045,0.070,0.000,2013-04,2026-11
5,10,S&P 500 TR USD vs CDT USD,204,0.711,-0.018,0.167,-0.052,0.121,0.054,2009-12,2026-11


In [10]:
display(Image('output/cdt_vs_sp500_ventanas_rodantes.png'))

(Imagen omitida en la versión de GitHub: ver output/*.png)


## 6. Verificación del "13 % vs 10 %"

In [11]:
print(json.dumps(r['res']['verificacion_amigo'], indent=1, ensure_ascii=False))

{
 "dtf_vigente": {
  "tasa_ea": 10.79,
  "vigencia": "2026-11-09..2026-11-15",
  "dtf_mensual_oct2026": 10.52
 },
 "sp500_precio_ytd_usd": 0.09866776360839236,
 "sp500_tr_ytd_usd": 0.10920565149902073,
 "sp500_tr_ytd_cop": -0.03463078355435123,
 "trm_31dic2025": 3757.08,
 "trm_corte": 3269.88,
 "sp500_tr_12m_usd": 0.11352063351893316,
 "cdt_ytd_cop_modelo": 0.08230819734984629,
 "inflacion_col_anual_oct2026": 0.06299420137058509,
 "sfc_cdt_360 días": {
  "promedio_ponderado_sistema": 12.236558043159759,
  "entidades": 40,
  "max_entidad": "RCI Colombia S.A. 13.30%",
  "pct_entidades_>=13": 0.225,
  "top5": {
   "RCI Colombia S.A.": 13.3,
   "Banco Pichincha S.A.": 13.28,
   "La Hipotecaria": 13.23,
   "\"KOA CF\"": 13.23,
   "Findeter": 13.22
  },
  "semana": "2026-11-03..2026-11-09"
 },
 "sfc_cdt_90 días": {
  "promedio_ponderado_sistema": 10.73694452356468,
  "entidades": 39,
  "max_entidad": "PLATA. S.A. COMPAÑÍA DE FINANCIAMIENTO 12.50%",
  "pct_entidades_>=13": 0.0,
  "top5": {
 

## 7. Empalme EEM → IEMG

In [12]:
r['res']['em_splice']

{'solape_desde': '2012-10-25',
 'solape_hasta': '2026-11-10',
 'n_dias': 3530,
 'corr_diaria': np.float64(0.9967),
 'corr_mensual': np.float64(0.9975),
 'dif_cagr_anual': np.float64(0.0063),
 'cagr_eem_solape': np.float64(0.0503),
 'cagr_iemg_solape': np.float64(0.0565)}